In [26]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
month = 202510
# 用于判断是否是新品
cur_date = '2025-10-31'
### 所需要的所有文件：
# 合并物流-财务-产品组-核算价-国内-用于低效-长尾 
# PLM的生命周期全表，因为涉及到生命周期，所以这里需要PLM的生命周期全表


### MAP信息汇总

In [27]:
Inefficient_standard_map = {
'吸油烟机':6000/12,
'灶具':6000/12,
'烤箱':1500/12,
'蒸箱':1500/12,
'微波炉':1500/12,
'蒸烤烹饪机':1500/12,
'蒸烤微烹饪机':1500/12,
'蒸微':1500/12,
'灶消烹饪机':1500/12,
'灶蒸烹饪机':1500/12,
'灶蒸烤烹饪机':1500/12,
'消毒柜':1000/12,
'热水器':900/12,
'两用炉':900/12,
'家用净水机':400/12,
'商用净水机':400/12,
'水槽洗碗机':2400/12,
'嵌入式洗碗机':2400/12,
}
productgroup_map={
    '吸油烟机': ['吸油烟机'],
    '灶具': ['灶具'],
    '蒸烤微合计': ['烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微'],
    '灶集成': ['灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机'],
    '消毒柜': ['消毒柜'],
    '热水器': ['热水器','两用炉'],
    '净水机': ['家用净水机','商用净水机'],
    '洗碗机': ['水槽洗碗机','嵌入式洗碗机']
}

### 读取单型号贡献报告中整理出的中间数据（处理了物流、财务、渠道、产品组、国内、核算价）

In [28]:
df = pd.read_excel(fr'D:\000物料报表\{month}\单型号贡献-低效-长尾\合并物流-财务-产品组-核算价-国内-用于低效-长尾.xlsx')
df['商品编码'] = df['商品编码'].astype(str).map(lambda x: x[:13])
df['渠道'].value_counts()
# print(len(df))

渠道
零售         328039
工程          14627
电商          13687
每誉            128
非零售工程电商        14
Name: count, dtype: int64

### 将产品的生命周期相关信息匹配进来

In [31]:
df_product_life = pd.read_excel(fr'D:\000物料报表\{month}\单型号贡献-低效-长尾\产品生命周期状态全表.xlsx')
# 转换物料号列为字符串类型，并只取前13位
df_product_life[['物料号']] = df_product_life[['物料号']].astype(str).applymap(lambda x: x[:13])
df_product_life = df_product_life[df_product_life['物料号'].str.len()>10]
df_product_life = df_product_life.drop_duplicates('物料号')
df_product_life_map_df = df_product_life[['物料号','产品状态','产品型号','开始销售时间']]


C:\Users\zhangbon\AppData\Local\Temp\ipykernel_22276\538384603.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_product_life[['物料号']] = df_product_life[['物料号']].astype(str).applymap(lambda x: x[:13])


In [32]:
df1 = pd.merge(df,df_product_life_map_df,how='left',left_on='商品编码',right_on='物料号')
df1['产品最早开始销售时间'] = df1.groupby('商品编码')['开始销售时间'].transform('min')


### 选出标准型号下面的型号存在量产阶段的标准型号,并只留下这一部分标准型号的数据（此时不涉及生命周期的筛选）

In [37]:
# 筛选掉新品（新品：最早开始销售时间到现在未满6个月的）
df_calu = df1[df1['产品最早开始销售时间']<=pd.Timestamp(cur_date)-pd.DateOffset(months=6)]
# 再在剩下的范围内选出有量产的标准型号
mass_standard = list(df_calu[df_calu['产品状态']=='量产']['标准型号'].drop_duplicates())
df_calu = df_calu[df_calu['标准型号'].isin(mass_standard)].reset_index(drop=True)

df_calu['各型号已售卖月份'] = ((pd.Timestamp(cur_date)-df_calu['产品最早开始销售时间']).dt.days/30).round()
df_calu['统计周期内已售卖月份'] = df_calu['各型号已售卖月份'].apply(lambda x: min(x, 12))
df_calu['各型号统计周期内总发货量'] = df_calu.groupby('产品型号')['实际出库数量'].transform('sum')
df_calu['各型号统计周期内月平均发货量'] = df_calu['各型号统计周期内总发货量']/df_calu['统计周期内已售卖月份']
df_calu = df_calu.drop_duplicates('物料号').reset_index(drop=True)
df_calu

,商品编码,渠道,实际出库数量,产品组,系统核算价,核算价,标准型号,国内/海外,物料号,产品状态,产品型号,开始销售时间,产品最早开始销售时间,各型号已售卖月份,统计周期内已售卖月份,各型号统计周期内总发货量,各型号统计周期内月平均发货量
0,1001002100022,工程,2,吸油烟机,1508,3016,JC03A,国内,1001002100022,量产,CXW-258-JC03A(不带罩),2021-05-17,2021-05-17,54.0,12.0,14405,1200.416667
1,1002003400032,工程,2,灶具,900,1800,TH3B,国内,1002003400032,退市预警,JZT-TH33B-12T,2021-09-01,2021-09-01,51.0,12.0,56372,4697.666667
2,1001002000018,工程,1,吸油烟机,3668,3668,03-X1A,国内,1001002000018,量产,CXW-358-03-X1A,2022-12-22,2022-12-22,35.0,12.0,57297,4774.750000
3,1003000500029,工程,1,消毒柜,1550,1550,ZTD100J-J31,国内,1003000500029,量产,ZTD100J-J51E,2021-04-30,2021-04-30,55.0,12.0,12977,1081.416667
4,1001001500097,工程,2,吸油烟机,2628,5256,Z7T,国内,1001001500097,量产,CXW-358-Z7T（不带罩）,2022-01-15,2022-01-15,46.0,12.0,234389,19532.416667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
636,1002001900418,工程,54,灶具,1730,93420,HECB,国内,1002001900418,量产,JZT-HECB(SH)-12T,2020-10-15,2020-10-15,61.0,12.0,54,4.500000
637,1002004300063,零售,5,灶具,940,4700,01-TD15B,国内,1002004300063,量产,JZT-01-TD15B-10T,2024-08-16,2024-08-16,15.0,12.0,5,0.416667
638,1003000500022,工程,157,消毒柜,1750,274750,ZTD100J-13T,国内,1003000500022,量产,ZTD100J-53GT,2018-12-30,2018-12-30,83.0,12.0,159,13.250000
639,1002003400029,电商,1,灶具,920,920,TH3B,国内,1002003400029,量产,JZY-TH15B-20Y,2018-09-01,2018-09-01,87.0,12.0,1,0.083333


### 计算出每个标准型号的总发货数，在依据其的产品组标准判断，各个各个标准型号是否是低效整机

In [38]:
df_calu['标准型号月均发货数'] = df_calu.groupby('标准型号')['各型号统计周期内月平均发货量'].transform('sum')
df_calu['标准型号是否低效'] = ''
for index, row in df_calu.iterrows():
    if row['标准型号月均发货数'] < Inefficient_standard_map[row['产品组']]:
        df_calu.loc[index,'标准型号是否低效'] = '是'
    else:
        df_calu.loc[index,'标准型号是否低效'] = '否'
df_calu

,商品编码,渠道,实际出库数量,产品组,系统核算价,核算价,标准型号,国内/海外,物料号,产品状态,产品型号,开始销售时间,产品最早开始销售时间,各型号已售卖月份,统计周期内已售卖月份,各型号统计周期内总发货量,各型号统计周期内月平均发货量,标准型号月均发货数,标准型号是否低效
0,1001002100022,工程,2,吸油烟机,1508,3016,JC03A,国内,1001002100022,量产,CXW-258-JC03A(不带罩),2021-05-17,2021-05-17,54.0,12.0,14405,1200.416667,1200.416667,否
1,1002003400032,工程,2,灶具,900,1800,TH3B,国内,1002003400032,退市预警,JZT-TH33B-12T,2021-09-01,2021-09-01,51.0,12.0,56372,4697.666667,30618.250000,否
2,1001002000018,工程,1,吸油烟机,3668,3668,03-X1A,国内,1001002000018,量产,CXW-358-03-X1A,2022-12-22,2022-12-22,35.0,12.0,57297,4774.750000,4774.750000,否
3,1003000500029,工程,1,消毒柜,1550,1550,ZTD100J-J31,国内,1003000500029,量产,ZTD100J-J51E,2021-04-30,2021-04-30,55.0,12.0,12977,1081.416667,5355.416667,否
4,1001001500097,工程,2,吸油烟机,2628,5256,Z7T,国内,1001001500097,量产,CXW-358-Z7T（不带罩）,2022-01-15,2022-01-15,46.0,12.0,234389,19532.416667,19532.416667,否
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
636,1002001900418,工程,54,灶具,1730,93420,HECB,国内,1002001900418,量产,JZT-HECB(SH)-12T,2020-10-15,2020-10-15,61.0,12.0,54,4.500000,780.166667,否
637,1002004300063,零售,5,灶具,940,4700,01-TD15B,国内,1002004300063,量产,JZT-01-TD15B-10T,2024-08-16,2024-08-16,15.0,12.0,5,0.416667,2835.083333,否
638,1003000500022,工程,157,消毒柜,1750,274750,ZTD100J-13T,国内,1003000500022,量产,ZTD100J-53GT,2018-12-30,2018-12-30,83.0,12.0,159,13.250000,231.333333,否
639,1002003400029,电商,1,灶具,920,920,TH3B,国内,1002003400029,量产,JZY-TH15B-20Y,2018-09-01,2018-09-01,87.0,12.0,1,0.083333,30618.250000,否


In [39]:
df_calu.to_excel(fr"C:\Users\zhangbon\Desktop\低效明细2.xlsx", index=False)

### 产品类别的低效标准型号、总标准型号数

In [40]:
df_calu2 = pd.DataFrame()
df_calu2['产品类别'] = productgroup_map.keys()
for k,v in productgroup_map.items():
    count_vals = df_calu[(df_calu['产品组'].isin(v))&(df_calu['标准型号是否低效']=='是')]['标准型号'].nunique()
    df_calu2.loc[df_calu2['产品类别'] == k,'低效标准型号数量'] = count_vals

    count_vals = df_calu[(df_calu['产品组'].isin(v))&(df_calu['标准型号是否低效']=='否')]['标准型号'].nunique()
    df_calu2.loc[df_calu2['产品类别'] == k,'非低效标准型号数量'] = count_vals
    
    count_vals = df_calu[(df_calu['产品组'].isin(v))]['标准型号'].nunique()
    df_calu2.loc[df_calu2['产品类别'] == k,'标准型号数量'] = count_vals
df_calu2


,产品类别,低效标准型号数量,非低效标准型号数量,标准型号数量
0,吸油烟机,26.0,71.0,97.0
1,灶具,13.0,37.0,50.0
2,蒸烤微合计,7.0,20.0,27.0
3,灶集成,6.0,13.0,19.0
4,消毒柜,5.0,17.0,22.0
5,热水器,15.0,28.0,43.0
6,净水机,6.0,10.0,16.0
7,洗碗机,23.0,33.0,56.0


### 统计各个渠道的低效产品型号数和产品型号数

In [10]:
df_calu3 = pd.DataFrame()
df_calu3['产品类别'] = productgroup_map.keys()
for k,v in productgroup_map.items():
    ### 零售
    count_vals1 = df_calu[(df_calu['产品组'].isin(v))&(df_calu['标准型号是否低效']=='是')&(df_calu['渠道']=='零售')]['产品型号'].nunique()
    df_calu3.loc[df_calu3['产品类别'] == k,'零售低效产品型号数量'] = count_vals1
    count_vals2 = df_calu[(df_calu['产品组'].isin(v))&(df_calu['渠道']=='零售')]['产品型号'].nunique()
    df_calu3.loc[df_calu3['产品类别'] == k,'零售产品型号数量'] = count_vals2
    df_calu3.loc[df_calu3['产品类别'] == k,'零售产品型号数量占比'] =  count_vals1/count_vals2

    ### 工程
    count_vals1 = df_calu[(df_calu['产品组'].isin(v))&(df_calu['标准型号是否低效']=='是')&(df_calu['渠道']=='工程')]['产品型号'].nunique()
    df_calu3.loc[df_calu3['产品类别'] == k,'工程低效产品型号数量'] = count_vals1
    count_vals2 = df_calu[(df_calu['产品组'].isin(v))&(df_calu['渠道']=='工程')]['产品型号'].nunique()
    df_calu3.loc[df_calu3['产品类别'] == k,'工程产品型号数量'] = count_vals2
    df_calu3.loc[df_calu3['产品类别'] == k,'工程产品型号数量占比'] =  count_vals1/count_vals2

    ### 电商
    count_vals1 = df_calu[(df_calu['产品组'].isin(v))&(df_calu['标准型号是否低效']=='是')&(df_calu['渠道']=='电商')]['产品型号'].nunique()
    df_calu3.loc[df_calu3['产品类别'] == k,'电商低效产品型号数量'] = count_vals1
    count_vals2 = df_calu[(df_calu['产品组'].isin(v))&(df_calu['渠道']=='电商')]['产品型号'].nunique()
    df_calu3.loc[df_calu3['产品类别'] == k,'电商产品型号数量'] = count_vals2
    df_calu3.loc[df_calu3['产品类别'] == k,'电商产品型号数量占比'] =  count_vals1/count_vals2

    ### 全渠道
    count_vals1 = df_calu[(df_calu['产品组'].isin(v))&(df_calu['标准型号是否低效']=='是')]['产品型号'].nunique()
    df_calu3.loc[df_calu3['产品类别'] == k,'全渠道低效产品型号数量'] = count_vals1
    count_vals2 = df_calu[(df_calu['产品组'].isin(v))]['产品型号'].nunique()
    df_calu3.loc[df_calu3['产品类别'] == k,'全渠道产品型号数量'] = count_vals2
    df_calu3.loc[df_calu3['产品类别'] == k,'全渠道产品型号数量占比'] =  count_vals1/count_vals2
df_calu3

,产品类别,零售低效产品型号数量,零售产品型号数量,零售产品型号数量占比,工程低效产品型号数量,工程产品型号数量,工程产品型号数量占比,电商低效产品型号数量,电商产品型号数量,电商产品型号数量占比,全渠道低效产品型号数量,全渠道产品型号数量,全渠道产品型号数量占比
0,吸油烟机,20.0,91.0,0.219780,16.0,72.0,0.222222,22.0,130.0,0.169231,43.0,182.0,0.236264
1,灶具,30.0,145.0,0.206897,11.0,63.0,0.174603,37.0,165.0,0.224242,51.0,240.0,0.212500
2,蒸烤微合计,6.0,24.0,0.250000,3.0,21.0,0.142857,10.0,38.0,0.263158,13.0,41.0,0.317073
3,灶集成,15.0,55.0,0.272727,7.0,17.0,0.411765,6.0,34.0,0.176471,20.0,60.0,0.333333
4,消毒柜,5.0,33.0,0.151515,3.0,24.0,0.125000,5.0,35.0,0.142857,6.0,46.0,0.130435
5,热水器,17.0,59.0,0.288136,1.0,21.0,0.047619,17.0,59.0,0.288136,21.0,77.0,0.272727
6,净水机,6.0,22.0,0.272727,1.0,10.0,0.100000,5.0,21.0,0.238095,12.0,31.0,0.387097
7,洗碗机,18.0,50.0,0.360000,7.0,28.0,0.250000,30.0,68.0,0.441176,37.0,79.0,0.468354


### 输出标准型号和渠道型号的统计分析

In [11]:

#输出df_qudao和df_calu2，写到一个excel里面
with pd.ExcelWriter(fr'D:\000物料报表\{month}\单型号贡献-低效-长尾\低效统计分析结果.xlsx') as writer:
    df_calu2.to_excel(writer, sheet_name='标准型号统计',index=False)
    df_calu3.to_excel(writer, sheet_name='分渠道产品型号统计',index=False)

